# FSL-AKE-IoD — Security Verification

Independent verification of the claimed BAKMM-IoD flaws (Phase 1), the adversarial review of FSL-AKE-IoD (Phase 2),
the formal-verification status (Phase 3), and a proof sketch.

Every result in this notebook comes from running `proposed/code/*.py`. Labels: **VERIFIED** (ran and passed),
**FAILED** (ran and the claim did not hold), **NOT RUN** (with the reason).

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "proposed", "code"))
import pandas as pd
pd.set_option("display.max_colwidth", 200)
from common import Clock, AuthError, h, xor, size_bits, reset_hash_counter, hashes
import attacks_fslake as A, attacks_bakmm_ext as X, fslake as F
IMPLS = A.impls()
def show(*attacks):
    rows = []
    for a in attacks:
        for i in IMPLS:
            r = a(i); rows.append(dict(attack=r.attack, protocol=r.protocol, outcome=r.outcome, reason=r.reason))
    return pd.DataFrame(rows)

**Output explanation.** There are four protocol columns: BAKMM-IoD with the ES updating TID on MSG3 (`BAKMM`) and on
MSG2 (`BAKMM[on_send]`), since the paper does not say which; FSL-AKE-IoD **v1** exactly as supplied with the draft
(`proposed_fslake.py`); and **v2** (`fslake.py`). The outcome is stated from the adversary's side: SUCCEEDED means the
attack works.

## Phase 0 — Equation diff: paper vs notebooks vs code vs .spdl

Formulas were read from rendered images of the paper (pp. 5–7), because the PDF text layer drops the math glyphs.
$A = h(TC_{DE}\parallel rs_1\parallel MS\parallel T_1)$, $B = h(RID_{ES}\parallel TC_{ES}\parallel rs_2\parallel MS\parallel T_2)$.

| Item | Paper (as printed) | Notebooks | `bakmm_iod.py` | `.spdl` models | Mismatch |
|---|---|---|---|---|---|
| M1 | $h(RID_{DE}\parallel MS\parallel T_1)\oplus A$ | same | same | same | – |
| M2 | $h(A\parallel RID_{DE}\parallel T_1)$ | same | same | same | – |
| M3 | $h(RID_{DE}\parallel MS\parallel T_2)\oplus B$ | same | same | same | – |
| **SK (ES, AKDDE2)** | $h(A\parallel B\parallel T_1\parallel T_2\parallel RID_{DE}\parallel MS)$ | same | same | same | – |
| **SK (DE, AKDDE3 + Table 5)** | $h(A\parallel B\parallel RID_{DE}\parallel T_1\parallel T_2\parallel MS)$ | **ES order** | **ES order** | **ES order** | **yes — printed protocol never agrees (A0)**; Attack_log item 1 is right |
| M4 | $h(SK\parallel T_1\parallel T_2\parallel RID_{DE})$ | same | same | same | – |
| M5 | $TID^{new}\oplus h(A\parallel RID_{DE}\parallel T_2)$ | same | same, hash truncated to 160 bit | same | 160-bit TID XOR 256-bit hash; §7.2 counts M5 as 256 bit |
| M6 | $h(SK\parallel T_3)$ | same | same | same | – |
| TID update at ES | "generates a new temporary identity"; **when** the ES overwrites is not stated | not stated | parameter `on_send` / `on_confirm` | not modelled | unspecified in paper |
| m1 | $h(TC_{ES}\parallel RS_1\parallel MS\parallel TS_1)\oplus h(RID_{ES}\parallel MS\parallel TS_1)$ | $h(RS_1\parallel TC_{ES}\ldots)$ (silently re-ordered) | **not implemented** | `03_*`: printed order | **inner order differs from m2 (E1)** |
| m2 | $h(h(RS_1\parallel TC_{ES}\parallel MS\parallel TS_1)\parallel RID_{ES}\parallel MS\parallel TS_1)$; CS check uses "$rs_1$" | same ($RS_1$) | not impl. | same | "rs1" typo (E2) |
| m3 | $h(RS_2\parallel TC_{CS}\parallel MS_{ES_j-ES_j}\parallel TS_2)\oplus h(RID_{ES}\parallel MS\parallel TS_1\parallel TS_2)$ | $MS_{ES-CS}$ | not impl. | $MS_{ES-CS}$ | "$MS_{ES_j-ES_j}$" typo (E3) |
| SK$_{ES,CS}$ | $h(h(RS_2\ldots)\parallel h(RS_1\ldots)\parallel RID_{ES}\parallel MS\parallel TS_1\parallel TS_2)$ | same | not impl. | same | – |
| m4 | $h(SK\parallel RID_{ES}\parallel MS\parallel TS_2)$ | uses $T_2$ (typo) | not impl. | same | notebook typo |
| m5 / TIN recovery | CS: $TIN^{new}\oplus h(RID_{ES}\parallel h(RS_1\parallel TC_{ES}\ldots)\parallel TS_2)$; **ES: $m_5\oplus h(RID_{ES}\parallel h(RS_2\parallel TC_{ES}\parallel MS\parallel TS_1)\parallel TS_2)$** | ES uses $RS_1$ (silently fixed) | not impl. | ES uses $RS_1$ | **ES cannot know $RS_2$ (E4)** |
| m6 | $h(SK\parallel TS_3)$ | same | not impl. | same | – |
| Root `bakmm_auth.spdl` | – | – | – | declares `rs1, rs2` both as `const` and as `fresh`/`var`; `T1..T3` as `const` | likely rejected by Scyther / timestamps not fresh |
| Root `bakmm_key_mgmt.spdl` | – | – | – | receives `var m3, m4, m5` unconstrained | checks not modelled (as `models/README.txt` warns) |
| All BAKMM `.spdl` | XOR | – | – | `const xor: Function` (uninterpreted) | intruder can never unmask; capture attack is invisible to Scyther |

**Draft (Week 3) vs paper.** The draft's §3.1 formulas match the ES-side order and do not mention the printed DE-side
order. Its "$|MSG_3| = 2880$" and "1782 instead of 1792" statements are confirmed (check **C**).

# Phase 1 — BAKMM-IoD claims re-checked

Threat model used (paper §3.2): Dolev–Yao (messages "can be accessed, modified, or deleted"), the CK model (session
states and keys can be revealed), and "$\mathcal{A}$ may also physically capture a certain number of drones and extract
data from their memory using an advanced power analysis method".

In [2]:
P1 = pd.DataFrame(X.run_all())
P1[["id", "title", "verdict"]]

,id,title,verdict
0,A0,Printed SK concatenation order differs between DE and ES,CONFIRMED
1,A0b,Correctness with the ES-side order used on both sides,CONFIRMED
2,A1,"Drone capture -> past and future SKs, TID chain",CONFIRMED
3,A2,Stolen ES verifier table -> impersonate every drone; TC_DE never checked,CONDITIONAL
4,A3,De-synchronisation by one dropped message (both ES update policies),CONFIRMED
5,A4,Replay of MSG1 inside dT,CONFIRMED (acceptance) / CONDITIONAL (lock-out)
6,A5,NEW: MSG2.M5 is not covered by M4 -> one flipped bit locks the drone out,CONFIRMED
7,AL3,Attack_log 3: replay after 2*dT,CONFIRMED (resisted as the log expects)
8,AL4,Attack_log 4: drone clock drift > dT (DoS),CONFIRMED
9,AL5,"Attack_log 5: ESL of rs1, rs2 only (CK)",CONFIRMED (resisted as the log expects)


In [3]:
for r in X.run_all():
    print(f"{r['id']}: {r['title']}\n   contradicts: {r['contradicts']}\n   assumption : {r['assumption']}\n   evidence   : {r['evidence']}\n")

A0: Printed SK concatenation order differs between DE and ES
   contradicts: Sec. 4.2 AKDDE2 vs AKDDE3 and Table 5 (spec erratum; Attack_log item 1)
   assumption : none (literal reading of the paper)
   evidence   : 0/1000 sessions agree on SK with the printed formulas (DE aborts at M4' = M4)

A0b: Correctness with the ES-side order used on both sides
   contradicts: -
   assumption : SK order harmonised (as bakmm_iod.py, the notebooks and all .spdl files do)
   evidence   : on_confirm 1000/1000, on_send 1000/1000 sessions agree

A1: Drone capture -> past and future SKs, TID chain
   contradicts: Prop. 5 ('would possess solely the session key and registration data of this particular drone'); ASFF8; ASFF13 (untraceability, Prop. 6); Sec. 3.2 ('supports forward secrecy'). NOT ASFF10: ESL with long-term keys intact still holds (see AL5)
   assumption : none: Sec. 3.2 lets A capture drones and extract memory by power analysis; RSDI2 stores {TID, RID, TC, MS} and no memory-encryption key i

**Output explanation (per check).**

* **A0** The paper's own SK formulas differ between the ES (AKDDE2) and the drone (AKDDE3 / Table 5), so with the
  printed formulas 0/1000 sessions agree. All of this repository's artefacts use the ES-side order on both sides (A0b,
  1000/1000). This is a specification erratum, not an attack.
* **A1 — CONFIRMED.** The masks $h(RID_{DE}\parallel MS\parallel T_x)$ depend only on static drone secrets and public
  timestamps, so capture gives every past and future SK and the whole TID chain (via M5). It contradicts Prop. 5,
  ASFF8, ASFF13 and §3.2 ("supports forward secrecy"). The draft also cites **ASFF10**; that is an over-reach, because
  ESL of $rs_1, rs_2$ alone is resisted (AL5).
* **A2 — CONDITIONAL.** It works as soon as the ES table is read, but the paper's Prop. 4 assumes access control
  prevents this, and §3.2 does not list ES compromise. "TC_DE is never checked" is unconditional: RSES2 does not even
  store $TC_{DE}$.
* **A3 — CONFIRMED** under both update readings (on_confirm: drop MSG3; on_send: drop MSG2). The paper makes no
  availability claim, but its DY model allows deletion.
* **A4 — acceptance CONFIRMED for on_confirm, REFUTED for on_send** (after MSG2 the old TID is already gone). The
  lock-out consequence also needs a single pending slot, which the paper does not specify.
* **A5 — NEW, CONFIRMED.** M4 authenticates SK but not M5. Flipping one bit of M5 makes the drone store a wrong
  TID while both sides report success, so the drone is locked out for good. This contradicts Prop. 2 ("it is not
  feasible for $\mathcal{A}$ to make any changes in the transmitted messages") and ASFF2.
* **E** The ES–CS phase as printed does not run: m1 and m2 use different inner-hash orders (the CS m2 check fails),
  and the ES's TIN recovery needs $RS_2$, which only the CS knows (session 1 succeeds, session 2 fails).
* **C** 704 + 800 + 288 = **1792** = Table 9. The paper's printed "2880" and "1782" are typos. The existing
  `bakmm_technical_analysis.tex` value of 1696 comes from counting M5 as 160 bits; it is not a paper error.

### Exact paper text for the errata (quoted from the rendered PDF)

* §7.2: "*$|MSG_1|$ = 160+256+256+32 = 704 bits, $|MSG_2|$ = 256+256+256+32 = 800 bits, and $|MSG_3|$ = 256+32 = 2880 bits, as a whole the communication of the BAKMM-IoD becomes 704+ 800+ 288 = 1782 bits.*" (Table 9: 1792.)
* AKDEC1: "*$m_1 = h(TC_{ES_j}\| RS_1\| MS_{ES_j-CS_k}\| TS_1)\oplus h(RID_{ES_j}\| MS_{ES_j-CS_k}\| TS_1)$ and $m_2 = h(h(RS_1\| TC_{ES_j}\| MS_{ES_j-CS_k}\| TS_1)\| RID_{ES_j}\| MS_{ES_j-CS_k}\| TS_1)$*"
* AKDEC2: "*$m'_2 = h(h(rs_1\| TC_{ES_j}\| MS_{ES_j-CS_k}\| TS_1)\| \ldots$*" and "*$m_3 = h(RS_2\| TC_{CS_k}\| MS_{ES_j-ES_j}\| TS_2)\oplus h(RID_{ES_j}\| MS_{ES_j-CS_k}\| TS_1\|TS_2)$*"
* AKDEC3: "*Further, $ES_j$ computes $TIN^{new}_{CS_k} = m_5 \oplus h(RID_{ES_j}\| h(RS_2\| TC_{ES_j}\| MS_{ES_j-CS_k}\| TS_1)\|$ $TS_2)$*"
* Other errata found: Prop. 1 is titled "*The SBBDA-IoD protocol makes it impossible to execute a replay attack*" (wrong scheme name); Table 10 marks BAKMM-IoD **×** for ASFF9 (formal verification) although §6 uses Scyther.

### Cross-check with `Attack_log.ipynb`

| Attack_log item | Covered by | Result here | Agreement |
|---|---|---|---|
| 1 SK concatenation-order mismatch | A0 | 0/1000 agree with the printed formulas | agrees |
| 2 De-sync by dropping MSG3 (ES commits on MSG3) | A3 | locked out; the on_send reading fails by dropping MSG2 instead | agrees, extended |
| 3 Replay after 2ΔT rejected | AL3 | rejected (stale T1) | agrees; the log misses the **in-window** replay (A4) |
| 4 Clock drift / GPS spoofing DoS | AL4, `delta_t_edges` | rejected beyond ΔT; affects FSL-AKE-IoD equally | agrees |
| 5 ESL of rs1, rs2 resisted | AL5 | 0/1000 guesses | agrees; but capture (A1) makes rs1/rs2 irrelevant |
| — | A1, A2, A4, **A5**, E, C | not in the log | added |

# Phase 2 — Trying to break FSL-AKE-IoD

Each subsection runs the PoC against all four protocol columns. Every PoC either succeeds or reports exactly why it
fails.

## 2.1 Replay — inside and outside ΔT, after resynchronisation, with clock skew, after an ES reboot

In [4]:
show(A.replay_inside, A.replay_outside, A.replay_after_resync, A.replay_with_clock_skew, A.reboot_replay)

,attack,protocol,outcome,reason
0,replay MSG1 inside dT,BAKMM,SUCCEEDED,same MSG1 accepted twice (no cache)
1,replay MSG1 inside dT,BAKMM[on_send],RESISTED,ES: unknown TID (desynchronised)
2,replay MSG1 inside dT,FSL-v1,RESISTED,ES: replay detected (cache)
3,replay MSG1 inside dT,FSL-v2,RESISTED,ES: replay detected (cache)
4,replay MSG1 outside dT,BAKMM,RESISTED,ES: stale T1
5,replay MSG1 outside dT,BAKMM[on_send],RESISTED,ES: stale T1
6,replay MSG1 outside dT,FSL-v1,RESISTED,ES: stale T1
7,replay MSG1 outside dT,FSL-v2,RESISTED,ES: stale T1
8,replay old MSG1 after resync,BAKMM,N/A,no resynchronisation mechanism
9,replay old MSG1 after resync,BAKMM[on_send],N/A,no resynchronisation mechanism


**Output explanation.** BAKMM-IoD accepts an in-window replay (on_confirm). v1's cache blocks the plain replay, but
it keeps entries only until *receive time* + ΔT. When the drone's clock runs 1 s ahead (legal, because |skew| < ΔT),
the message is still fresh after the entry has expired. The replay is then accepted, and because the replay matches
v1's *old* record, the ES deletes the record the drone holds: **permanent lock-out**. The same happens after an ES
reboot empties the cache. v2 keeps entries until $T_1+\Delta T$ (F2) and imposes a 2ΔT quiet period after a reboot.

## 2.2 MITM / message modification, reflection, parallel-session interleaving

In [5]:
show(A.modify_fields, A.reflection, A.interleave)

,attack,protocol,outcome,reason
0,MITM: modify one field,BAKMM,SUCCEEDED,"accepted modified ['MSG2.M5 (session completes, then drone locked out=True)']"
1,MITM: modify one field,BAKMM[on_send],SUCCEEDED,"accepted modified ['MSG2.M5 (session completes, then drone locked out=True)']"
2,MITM: modify one field,FSL-v1,RESISTED,every single-field modification of MSG1 and MSG2 rejected
3,MITM: modify one field,FSL-v2,RESISTED,every single-field modification of MSG1 and MSG2 rejected
4,reflection,BAKMM,RESISTED,MSG1/MSG2 have different structure and keyed checks
5,reflection,BAKMM[on_send],RESISTED,MSG1/MSG2 have different structure and keyed checks
6,reflection,FSL-v1,RESISTED,MSG1/MSG2 have different structure and keyed checks
7,reflection,FSL-v2,RESISTED,MSG1/MSG2 have different structure and keyed checks
8,parallel-session interleaving,BAKMM,RESISTED,MSG2 is bound to the drone's key and N1/T1
9,parallel-session interleaving,BAKMM[on_send],RESISTED,MSG2 is bound to the drone's key and N1/T1


**Output explanation.** Every field of both FSL messages is covered by V1 or V2, so each single-field change is
rejected. In BAKMM-IoD, **M5 is not covered by M4** (new flaw A5): the modified M5 is accepted and the drone is then
locked out. Reflection fails because MSG1 and MSG2 have different structures and keyed checks. Swapped MSG2s
between two parallel drones fail because MSG2 is bound to the drone's own K, N1 and T1.

## 2.3 Drone impersonation and ES impersonation (no secrets)

In [6]:
show(A.drone_impersonation, A.es_impersonation)

,attack,protocol,outcome,reason
0,drone impersonation (no secrets),BAKMM,RESISTED,3 forged MSG1 (observed fields + current TID + fresh T) rejected
1,drone impersonation (no secrets),BAKMM[on_send],RESISTED,3 forged MSG1 (observed fields + current TID + fresh T) rejected
2,drone impersonation (no secrets),FSL-v1,RESISTED,3 forged MSG1 (observed fields + current TID + fresh T) rejected
3,drone impersonation (no secrets),FSL-v2,RESISTED,3 forged MSG1 (observed fields + current TID + fresh T) rejected
4,ES impersonation (no secrets),BAKMM,RESISTED,"old MSG2 with fresh T2, and random MSG2, both rejected"
5,ES impersonation (no secrets),BAKMM[on_send],RESISTED,"old MSG2 with fresh T2, and random MSG2, both rejected"
6,ES impersonation (no secrets),FSL-v1,RESISTED,"old MSG2 with fresh T2, and random MSG2, both rejected"
7,ES impersonation (no secrets),FSL-v2,RESISTED,"old MSG2 with fresh T2, and random MSG2, both rejected"


**Output explanation.** Neither side can be impersonated without K (FSL) or MS/RID (BAKMM): recombined observed fields and random messages are all rejected.

## 2.4 Drone capture — KCI, past sessions (forward secrecy), future sessions, linkability of past TIDs

In [7]:
show(A.kci_after_capture, A.capture_past_keys, A.capture_future_keys, A.capture_link_tids)

,attack,protocol,outcome,reason
0,KCI after drone capture,BAKMM,SUCCEEDED,drone accepts forged MSG2 and shares SK with the adversary (inherent to symmetric-only AKE)
1,KCI after drone capture,BAKMM[on_send],SUCCEEDED,drone accepts forged MSG2 and shares SK with the adversary (inherent to symmetric-only AKE)
2,KCI after drone capture,FSL-v1,SUCCEEDED,drone accepts forged MSG2 and shares SK with the adversary (inherent to symmetric-only AKE)
3,KCI after drone capture,FSL-v2,SUCCEEDED,drone accepts forged MSG2 and shares SK with the adversary (inherent to symmetric-only AKE)
4,capture: past SKs (forward secrecy),BAKMM,SUCCEEDED,5/5 past session keys recovered from captured memory
5,capture: past SKs (forward secrecy),BAKMM[on_send],SUCCEEDED,5/5 past session keys recovered from captured memory
6,capture: past SKs (forward secrecy),FSL-v1,RESISTED,0/5 past keys: captured K_n = h(K_(n-1)||SK_(n-1)) cannot be inverted
7,capture: past SKs (forward secrecy),FSL-v2,RESISTED,0/5 past keys: captured K_n = h(K_(n-1)||SK_(n-1)) cannot be inverted
8,capture: future SKs,BAKMM,SUCCEEDED,3/3 future keys by passively following the chain (no post-compromise security)
9,capture: future SKs,BAKMM[on_send],SUCCEEDED,3/3 future keys by passively following the chain (no post-compromise security)


**Output explanation.**
* **Forward secrecy holds** in FSL: the captured $K_n$ cannot be inverted to $K_{n-1}$, so 0/5 past keys are recovered.
* **KCI succeeds** in both protocols. With symmetric keys only, whoever holds the drone's key can impersonate the ES
  *to that drone*. This is inherent to the design and should be stated as a limitation rather than left unclaimed.
* **Future sessions** fall in both protocols: the adversary follows the key chain passively (no post-compromise
  security). The draft's Table 3 acknowledges this.
* **Linkability of past TIDs** in FSL is PARTIAL. After successful sessions nothing links. But if the drone's last
  attempt failed (MSG2 dropped), its stored TID equals the TID last seen on the air, which links that attempt.

## 2.5 ES database — stolen without $X_{ES}$, and full ES compromise with $X_{ES}$ (quantified)

In [8]:
show(A.stolen_db_without_X, A.es_compromise_with_X)

,attack,protocol,outcome,reason
0,stolen ES DB (no X_ES),BAKMM,SUCCEEDED,"plain (RID, MS) table: forged drone accepted, adversary shares SK; random TC_DE accepted"
1,stolen ES DB (no X_ES),BAKMM[on_send],SUCCEEDED,"plain (RID, MS) table: forged drone accepted, adversary shares SK; random TC_DE accepted"
2,stolen ES DB (no X_ES),FSL-v1,RESISTED,C = K xor h(X_ES||TID) is useless without X_ES
3,stolen ES DB (no X_ES),FSL-v2,RESISTED,C = K xor h(X_ES||TID) is useless without X_ES
4,ES compromise incl. X_ES,BAKMM,SUCCEEDED,same as stolen DB: every past and future session of every drone (MS is static)
5,ES compromise incl. X_ES,BAKMM[on_send],SUCCEEDED,same as stolen DB: every past and future session of every drone (MS is static)
6,ES compromise incl. X_ES,FSL-v1,PARTIAL,"3/15 past sessions exposed (the last one per drone, via the kept old record); 3/3 next sessions exposed (all future sessions until re-keying)"
7,ES compromise incl. X_ES,FSL-v2,PARTIAL,"3/15 past sessions exposed (the last one per drone, via the kept old record); 3/3 next sessions exposed (all future sessions until re-keying)"


**Output explanation.** Without $X_{ES}$ the masked table is useless. With $X_{ES}$ (3 drones × 5 sessions),
**3/15 past sessions** are exposed: exactly the last one of each drone, because the ES keeps the anchor key until the
drone's next session. All future sessions until re-keying are exposed as well. This matches the draft's "only the most
recent session key" claim. The ablation notebook shows that adding MSG3 (M6 off) removes even that single session.

## 2.6 ESL / CK — reveal N1, N2 only; reveal SK_n only (does it give K_n?); reveal state mid-session; known-session-key

In [9]:
show(A.esl_ephemerals_only, A.reveal_sk_get_k, A.reveal_state_mid_session, A.known_session_key)

,attack,protocol,outcome,reason
0,ESL: reveal ephemerals only,BAKMM,RESISTED,"rs1, rs2 alone do not give A, B or SK (all need MS)"
1,ESL: reveal ephemerals only,BAKMM[on_send],RESISTED,"rs1, rs2 alone do not give A, B or SK (all need MS)"
2,ESL: reveal ephemerals only,FSL-v1,RESISTED,"N1, N2 are public anyway; SK needs K (claim is true but vacuous: nonces carry no secrecy)"
3,ESL: reveal ephemerals only,FSL-v2,RESISTED,"N1, N2 are public anyway; SK needs K (claim is true but vacuous: nonces carry no secrecy)"
4,reveal SK_n -> long-term key?,BAKMM,RESISTED,SK is a hash output; MS not derivable
5,reveal SK_n -> long-term key?,BAKMM[on_send],RESISTED,SK is a hash output; MS not derivable
6,reveal SK_n -> long-term key?,FSL-v1,RESISTED,K_(n+1) = h(K_n||SK_n) still needs K_n (preimage of SHA-256)
7,reveal SK_n -> long-term key?,FSL-v2,RESISTED,K_(n+1) = h(K_n||SK_n) still needs K_n (preimage of SHA-256)
8,reveal session state mid-run,BAKMM,RESISTED,"state {rs1, T1, A, B}: SK = h(A||B||T1||T2||RID||MS) still needs RID and MS"
9,reveal session state mid-run,BAKMM[on_send],RESISTED,"state {rs1, T1, A, B}: SK = h(A||B||T1||T2||RID||MS) still needs RID and MS"


**Output explanation.** All four are resisted. Note that the FSL ESL claim is **vacuous**: $N_1, N_2$ are sent in the
clear, so ephemeral leakage reveals nothing new, and all secrecy rests on $K$. Revealing $SK_n$ does not give
$K_{n+1}=h(K_n\parallel SK_n)$ without $K_n$. Revealed keys do not help with other sessions.

## 2.7 De-synchronisation — drop MSG2 once, repeatedly, reordered retries across the two records, eviction, two ES

In [10]:
show(A.desync_drop_once, A.desync_drop_repeated, A.desync_reordered_retry, A.desync_candidate_eviction, A.cross_es)

,attack,protocol,outcome,reason
0,de-sync: drop one message,BAKMM,SUCCEEDED,drop MSG3 -> locked out (ES: unknown TID (desynchronised))
1,de-sync: drop one message,BAKMM[on_send],SUCCEEDED,drop MSG2 -> locked out (ES: unknown TID (desynchronised))
2,de-sync: drop one message,FSL-v1,RESISTED,drop MSG2 -> recovered next run
3,de-sync: drop one message,FSL-v2,RESISTED,drop MSG2 -> recovered next run
4,de-sync: drop MSG2 10x,BAKMM,RESISTED,after 10 drops the drone still authenticates
5,de-sync: drop MSG2 10x,BAKMM[on_send],SUCCEEDED,ES: unknown TID (desynchronised)
6,de-sync: drop MSG2 10x,FSL-v1,RESISTED,after 10 drops the drone still authenticates
7,de-sync: drop MSG2 10x,FSL-v2,RESISTED,after 10 drops the drone still authenticates
8,de-sync: reordered retry (2 records),BAKMM,RESISTED,"withheld MSG1(a) accepted=False, but the drone's record is kept"
9,de-sync: reordered retry (2 records),BAKMM[on_send],RESISTED,"withheld MSG1(a) accepted=False, but the drone's record is kept"


**Output explanation.**
* Single and repeated drops are handled by both FSL versions.
* **v1 breaks across its two records.** The drone retries (two fresh MSG1 on the same credentials). The adversary
  delivers the second first, so the drone completes with it, and then releases the first. v1 treats that as a message
  on the *old* record and deletes the *current* record, which is the one the drone holds. With six withheld retries the
  same happens. v2's candidate set (F3) and retry gap (F4) close both.
* A drone provisioned at **two ES** with the same credentials stops working at the second ES as soon as it rotates at
  the first (BAKMM-IoD too). Roaming needs per-ES credentials. This is a functional limitation, not an attack.

## 2.8 DoS — cache flooding, TID lookup exhaustion, clock skew and ΔT edges

In [11]:
show(A.dos_costs, A.delta_t_edges)

,attack,protocol,outcome,reason
0,DoS: TID lookup / cache flooding,BAKMM,RESISTED,"unknown TID costs 0 h, known TID + garbage costs 2 h; cache entries after 400 forgeries = 0 (inserted only after a valid MAC)"
1,DoS: TID lookup / cache flooding,BAKMM[on_send],RESISTED,"unknown TID costs 0 h, known TID + garbage costs 2 h; cache entries after 400 forgeries = 0 (inserted only after a valid MAC)"
2,DoS: TID lookup / cache flooding,FSL-v1,RESISTED,"unknown TID costs 0 h, known TID + garbage costs 2 h; cache entries after 400 forgeries = 0 (inserted only after a valid MAC)"
3,DoS: TID lookup / cache flooding,FSL-v2,RESISTED,"unknown TID costs 0 h, known TID + garbage costs 2 h; cache entries after 400 forgeries = 0 (inserted only after a valid MAC)"
4,clock skew / dT edges,BAKMM,PARTIAL,"|skew|<=dT accepted, >dT rejected {-3: False, -2: True, 2: True, 3: False}: a drone whose clock drifts > dT is locked out until resynchronised (inherent to timestamp freshness)"
5,clock skew / dT edges,BAKMM[on_send],PARTIAL,"|skew|<=dT accepted, >dT rejected {-3: False, -2: True, 2: True, 3: False}: a drone whose clock drifts > dT is locked out until resynchronised (inherent to timestamp freshness)"
6,clock skew / dT edges,FSL-v1,PARTIAL,"|skew|<=dT accepted, >dT rejected {-3: False, -2: True, 2: True, 3: False}: a drone whose clock drifts > dT is locked out until resynchronised (inherent to timestamp freshness)"
7,clock skew / dT edges,FSL-v2,PARTIAL,"|skew|<=dT accepted, >dT rejected {-3: False, -2: True, 2: True, 3: False}: a drone whose clock drifts > dT is locked out until resynchronised (inherent to timestamp freshness)"


**Output explanation.** An unknown TID costs the ES 0 hashes, and a known TID with garbage costs 2. Cache entries are
inserted only after a valid MAC, so forged messages cannot fill the cache. $|skew| = \Delta T$ is accepted and
$\Delta T+1$ rejected (inclusive bound). A drone whose clock drifts beyond ΔT is locked out until its clock is
corrected. That is inherent to timestamp freshness and applies to both protocols.

## 2.9 Privileged insider at the RA; dynamic addition and revocation

In [12]:
show(A.insider_ra, A.dynamic_add_default_enroll, A.revocation)

,attack,protocol,outcome,reason
0,privileged insider at RA,BAKMM,PARTIAL,if the RA keeps RID/MS (Prop. 3 assumes it deletes them) every SK is recoverable
1,privileged insider at RA,BAKMM[on_send],PARTIAL,if the RA keeps RID/MS (Prop. 3 assumes it deletes them) every SK is recoverable
2,privileged insider at RA,FSL-v1,PARTIAL,"if the RA keeps K_0: 6/6 SKs with every transcript, 2/6 when one transcript is missed (the chain breaks); requires RA to erase K_i"
3,privileged insider at RA,FSL-v2,PARTIAL,"if the RA keeps K_0: 6/6 SKs with every transcript, 2/6 when one transcript is missed (the chain breaks); requires RA to erase K_i"
4,dynamic addition (2nd drone),BAKMM,RESISTED,both drones keep working
5,dynamic addition (2nd drone),BAKMM[on_send],RESISTED,both drones keep working
6,dynamic addition (2nd drone),FSL-v1,SUCCEEDED,drone 1's session deleted drone 2's record (ES: unknown TID): enroll(dev='DE') default
7,dynamic addition (2nd drone),FSL-v2,RESISTED,both drones keep working
8,revocation,BAKMM,N/A,no revocation procedure in the paper
9,revocation,BAKMM[on_send],N/A,no revocation procedure in the paper


**Output explanation.**
* **Insider (PARTIAL for all).** If the RA keeps $K_0$ and records *every* transcript, it derives all SKs (6/6). If it
  misses one transcript, the chain breaks (only the 2 earlier SKs). The RA must erase $K_i$, which is the same
  assumption BAKMM-IoD's Prop. 3 makes. Only a public-key exchange would remove it.
* **Dynamic addition in v1 is broken.** `enroll()` defaults to `dev="DE"`, so the first drone's session deletes the
  second drone's record (**F1**).
* **Revocation.** Deleting only the *current* record (v1) leaves the old record, and a drone whose MSG2 was dropped
  still authenticates. v2's `revoke(dev)` removes the anchor and all candidates.

## 2.10 TID collision after truncating $h(\cdot)$ to 160 bits

Demo with 8-bit TIDs to force collisions, plus the analytic bound for 160 bits.

In [13]:
import cost
display(show(A.tid_collision))
b = cost.tid_collision_bound(); print(f"160-bit bound: G = {b['pseudonyms']:.1e} pseudonyms -> P(collision) <= G^2 / 2^161 = {b['bound']:.2e}")

,attack,protocol,outcome,reason
0,TID collision (8-bit demo),BAKMM,N/A,ES picks TID_new at random; can check uniqueness
1,TID collision (8-bit demo),BAKMM[on_send],N/A,ES picks TID_new at random; can check uniqueness
2,TID collision (8-bit demo),FSL-v1,SUCCEEDED,42/160 sessions failed with 8-bit TIDs (dict keyed by TID: colliding record overwritten)
3,TID collision (8-bit demo),FSL-v2,RESISTED,0/160 sessions failed with 8-bit TIDs (ES tries every record under a TID)


160-bit bound: G = 2.0e+12 pseudonyms -> P(collision) <= G^2 / 2^161 = 1.37e-24


**Output explanation.** v1 indexes records in a dict keyed by TID, so a collision overwrites another drone's record
(47/160 sessions fail with 8-bit TIDs). v2 keeps a list per identifier and tries each record's key, so collisions are
harmless (0/160). At 160 bits, even $10^6$ drones × $10^6$ sessions give $P\le1.4\times10^{-24}$.

## 2.11 Anonymity — can a passive observer link two sessions of the same drone?

In [14]:
display(show(A.anonymity_passive))
print(A.anonymity_passive(A.FSLv2Impl(F.without(retry_pseudonyms=3), "FSL-v2 + retry pseudonyms (w=3)")))

,attack,protocol,outcome,reason
0,anonymity: passive linking,BAKMM,PARTIAL,successful sessions unlinkable; each dropped MSG2 makes the retry reuse the TID (4 linkable pairs in 12); an active adversary dropping every MSG2 can track the drone
1,anonymity: passive linking,BAKMM[on_send],PARTIAL,successful sessions unlinkable; each dropped MSG2 makes the retry reuse the TID (10 linkable pairs in 12); an active adversary dropping every MSG2 can track the drone
2,anonymity: passive linking,FSL-v1,PARTIAL,successful sessions unlinkable; each dropped MSG2 makes the retry reuse the TID (4 linkable pairs in 12); an active adversary dropping every MSG2 can track the drone
3,anonymity: passive linking,FSL-v2,PARTIAL,successful sessions unlinkable; each dropped MSG2 makes the retry reuse the TID (4 linkable pairs in 12); an active adversary dropping every MSG2 can track the drone


[RESISTED ] anonymity: passive linking         FSL-v2 + retry pseudonyms (w=3) no identifier repeats on the air


**Output explanation.** Successful sessions are unlinkable in both protocols. But **every failed attempt makes the
retry reuse the same TID**: 4 linkable pairs in 12 sessions with one third of MSG2 dropped. An active adversary who drops
every MSG2 can therefore track a drone indefinitely. The draft's "an eavesdropper cannot link sessions" is an overclaim.
The optional extension `retry_pseudonyms = w` (the drone sends $h(TID\parallel j)$ on retry $j$) removes it for up to
$w$ retries, at a cost of $w$ extra hashes per session on the ES (7 → 10 for $w=3$).

# ATTACK × PROTOCOL matrix

In [15]:
M = A.matrix()
names = [i.name for i in IMPLS]
mat = pd.DataFrame([{**{"attack": row[names[0]].attack}, **{n: row[n].outcome for n in names},
                     "FSL-v2 reason": row["FSL-v2"].reason} for row in M.values()])
mat

,attack,BAKMM,BAKMM[on_send],FSL-v1,FSL-v2,FSL-v2 reason
0,replay MSG1 inside dT,SUCCEEDED,RESISTED,RESISTED,RESISTED,ES: replay detected (cache)
1,replay MSG1 outside dT,RESISTED,RESISTED,RESISTED,RESISTED,ES: stale T1
2,replay old MSG1 after resync,N/A,N/A,RESISTED,RESISTED,replay of MSG1(a): ES: stale T1; drone still works
3,replay + clock skew (+1 s),RESISTED,RESISTED,SUCCEEDED,RESISTED,ES: replay detected (cache)
4,replay after ES reboot (cache lost),N/A,N/A,SUCCEEDED,RESISTED,ES: quiet period after reboot
5,MITM: modify one field,SUCCEEDED,SUCCEEDED,RESISTED,RESISTED,every single-field modification of MSG1 and MSG2 rejected
6,reflection,RESISTED,RESISTED,RESISTED,RESISTED,MSG1/MSG2 have different structure and keyed checks
7,parallel-session interleaving,RESISTED,RESISTED,RESISTED,RESISTED,MSG2 is bound to the drone's key and N1/T1
8,drone impersonation (no secrets),RESISTED,RESISTED,RESISTED,RESISTED,3 forged MSG1 (observed fields + current TID + fresh T) rejected
9,ES impersonation (no secrets),RESISTED,RESISTED,RESISTED,RESISTED,"old MSG2 with fresh T2, and random MSG2, both rejected"


In [16]:
mat[names].apply(pd.Series.value_counts).fillna(0).astype(int)

,BAKMM,BAKMM[on_send],FSL-v1,FSL-v2
N/A,4,4,0,0
PARTIAL,4,4,6,6
RESISTED,15,15,17,24
SUCCEEDED,9,9,9,2


**Output explanation.** Against FSL-AKE-IoD **v1 as drafted, 9 of 32 attacks succeed**. Seven of them are
permanent lock-outs or bugs that v2 fixes; the other two (KCI and future keys after capture) are inherent.
**v2 leaves only** the rows that no symmetric-key-only protocol can resist (KCI after capture, future keys after
capture) plus PARTIAL rows: one past session exposed on full ES compromise, linkability of failed attempts and of the
last failed attempt after capture, the RA-insider assumption, clock drift beyond ΔT, and roaming between ground
stations. BAKMM-IoD fails every capture and database row, one-message de-sync,
and the M5 modification.

# Phase 3 — Formal verification status

| Model | Purpose | Expected result | Status |
|---|---|---|---|
| `bakmm_auth.spdl`, `bakmm_key_mgmt.spdl`, `models/*.spdl` | existing BAKMM models | as in the paper (Fig. 5) | **NOT RUN** — Scyther not installed |
| `proposed/code/fslake_iod.spdl` | draft's FSL model | Secret/SKR/Alive/Weakagree Ok; the draft did not claim ES Niagree | **NOT RUN** |
| `fslake_auth.spdl`, `fslake_key_mgmt.spdl` | v2 DE–ES / ES–CS | DE claims Ok; ES Niagree → replay trace (cache not expressible) | **NOT RUN** |
| `bakmm_auth_compromise.spdl` | capture after session, XOR as encryption | **Secret SK Fail** | **NOT RUN** (Python equivalent A1 VERIFIED) |
| `bakmm_auth_compromise_xorfun.spdl` | same, XOR as uninterpreted function (paper style) | Secret SK **Ok = false negative** | **NOT RUN** |
| `fslake_auth_compromise.spdl` | capture after session, evolved key leaks | **Secret SK Ok** | **NOT RUN** (Python `capture_past_keys` VERIFIED) |
| `fslake_auth_compromise_control.spdl` | control: un-evolved key leaks | Secret SK Fail | **NOT RUN** |
| `proverif/bakmm_fs.pv`, `proverif/fslake_fs.pv` | phase-1 capture, `query attacker(secret)` | BAKMM: attacker true; FSL: not attacker | **NOT RUN** — ProVerif not installed |

Run everything with `SCYTHER=/path/to/scyther-linux bash scripts/run_formal.sh`. It writes `formal_results/*.txt`
and claim-table PNGs to `images/scyther_*.png`.

**Important modelling point.** The paper's Scyther model (Fig. 3/4), and every BAKMM `.spdl` in this repository,
declares `const xor: Function;`, an *uninterpreted* symbol with no cancellation law. The intruder can then never
remove a mask, so Scyther **cannot** find A1, however many runs it explores. The paper's "No attacks within bounds"
is therefore no evidence of capture resistance. `bakmm_auth_compromise.spdl` models masking as encryption keyed by
the mask, which is sound for this purpose.

# Proof sketch (ROR model) for FSL-AKE-IoD v2

**Model.** A Real-or-Random game in the style of Abdalla–Fouque–Pointcheval. $h$ is a random oracle with output
length $\ell=256$ ($\ell_T=160$ for TIDs). $\mathcal{A}$ has $Execute$, $Send$, $Reveal$ (session key),
$Corrupt_{DE}$ (the drone's current $\{TID,K\}$), $Corrupt_{ES}$ (DB without $X_{ES}$) and one $Test$ query.
Freshness excludes: Reveal of the tested session or its partner; Corrupt of the tested drone *before* the tested
session completes (so forward secrecy covers corruption *after*); and Corrupt of $X_{ES}$ before the drone's next
session. Let $q_h$ be the number of hash queries, $q_s$ the number of Send queries and $q_e$ the number of Execute
queries.

**Game sequence.**
* $G_0$: the real protocol. $Adv = |2\Pr[Succ_0]-1|$.
* $G_1$: simulate $h$ with a table. Identical: $\Pr[Succ_1]=\Pr[Succ_0]$.
* $G_2$: abort on collisions of $h$ outputs, nonces or TIDs. $|\Pr[Succ_2]-\Pr[Succ_1]| \le \frac{q_h^2}{2^{\ell+1}} + \frac{(q_s+q_e)^2}{2^{161}}$.
* $G_3$: abort if $\mathcal{A}$ produces a valid $V_1$ or $V_2$ without querying $h$ on $K$. Forging a MAC-like tag
  without the key costs $\le q_s/2^{\ell}$. After this, entity authentication holds; with the cache (F2) and the
  quiet period, each accepted $MSG_1$ is fresh, which gives injectivity.
* $G_4$: replace $SK$ in the tested session by a random value. $\mathcal{A}$ notices only by querying
  $h(K_n\parallel N_1\parallel N_2\parallel T_1\parallel T_2\parallel TID)$ with the right $K_n$. Before the test
  session, $K_n$ is uniformly random given the view, because each $K_j=h(K_{j-1}\parallel SK_{j-1})$ is a fresh
  random-oracle output and $SK_{j-1}$ is hidden by $G_3$. After a later $Corrupt_{DE}$, $\mathcal{A}$ holds
  $K_{n+1}=h(K_n\parallel SK_n)$, which is independent of $K_n$ in the ROM. $Corrupt_{ES}$ without $X_{ES}$ gives
  $C=K\oplus h(X_{ES}\parallel TID)$, a one-time pad. $|\Pr[Succ_4]-\Pr[Succ_3]|\le q_h/2^{\ell}$, and $\Pr[Succ_4]=1/2$.

$$Adv^{ROR}_{\text{FSL-AKE}}(\mathcal{A}) \le \frac{q_h^2}{2^{\ell}} + \frac{(q_s+q_e)^2}{2^{160}} + \frac{2q_s+2q_h}{2^{\ell}}.$$

**What the proof does *not* give.** KCI (a corrupted drone key authenticates anything *to* that drone); security of
sessions after a corruption (no post-compromise security); availability. Those rows are confirmed empirically in the
matrix above. This is a sketch, not a machine-checked proof; the ProVerif query in `proverif/fslake_fs.pv` is the
automated counterpart (NOT RUN).